# Managing data

**Optional depth track · module 2 of 5**

**Goal:** Move the corpus out of memory and into a store you can correct, version and delete from. Then say what you refuse to keep.

**Why it matters:** Data is the one layer an agent cannot refactor away later. The article calls it relatively hard to change, and it is right: an answer given last month is only explainable if you can still say which version of which document it was grounded in. That is a schema decision, and it is cheap now and expensive in six months.

Nothing here is graded and nothing in the fifteen sessions depends on it. Work through it when you want the layer underneath.

In [ ]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
    print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

In [ ]:
# depth_checks registers this track's checkers. check() and review() are the
# same ones the course uses.
from bootcamp_agent.checks import check, review
import bootcamp_agent.depth_checks  # noqa: F401

## 1. Choose the store, from the access pattern

**Context.** Six Markdown files load into memory today. That is a real choice and it is fine
until it is not. Argue the next one from how the data is actually used, not from what you
like writing.

**Instructions.**

1. Name the store you would use for the corpus and its versions.
2. Write the access pattern first: what reads it, how often, what writes it, and when.
3. Name what the choice costs. Every store makes something harder, and if you cannot say
   what, you have not chosen it, you have defaulted to it.

In [ ]:
choice = {
    "store": "sqlite, one row per document version",
    "access_pattern": "",   # TODO(you): reads and writes, how often, by whom
    "why": "",              # TODO(you): connect the pattern to the store
    "what_it_costs": "",    # TODO(you): what this makes harder
}
for k, v in choice.items():
    print(f"{k:16} {v or '(empty)'}")

**Expected output**

```
store            sqlite, one row per document version
access_pattern   Read on every question, whole corpus, hundreds of times an hour. ...
why              Read-heavy, tiny, single-writer, and it has to survive a restart. ...
what_it_costs    One file to back up and migrate, ...
✅ d2-e1 passed
```

In [ ]:
check("d2-e1", choice)

## 2. Ingestion that can be run twice

**Context.** Every import script gets run twice: once by you, once by cron, once by the person
who did not know you had already run it. If the second run duplicates the corpus, retrieval
starts scoring the same passage twice.

**Instructions.**

1. Create a `documents` table with `doc_id`, `content_hash`, `text`.
2. Hash the text. If that exact `(doc_id, content_hash)` is already there, do nothing.
3. If the text **changed**, insert a new row. A correction is a new version, not an overwrite,
   or you can no longer say what an old answer was grounded in.

In [ ]:
import hashlib
import sqlite3


def ingest(conn: sqlite3.Connection, doc_id: str, text: str) -> None:
    conn.execute(
        "create table if not exists documents ("
        " doc_id text, content_hash text, text text,"
        " primary key (doc_id, content_hash))"
    )
    # TODO(you): hash the text, then insert only if this exact version is new.
    conn.commit()


conn = sqlite3.connect(":memory:")
ingest(conn, "rag-basics", "chunking splits a document")
ingest(conn, "rag-basics", "chunking splits a document")
ingest(conn, "rag-basics", "chunking splits a document into passages")
print("rows:", conn.execute("select count(*) from documents").fetchone()[0])

**Expected output**

```
rows: 2
✅ d2-e2 passed
```

Two rows, not three and not one: the repeat was ignored, the correction became a version.

In [ ]:
check("d2-e2", ingest)

## 3. The retention policy

**Context.** A policy that keeps everything is not a policy. The useful part is the list of
things you refuse to store at all, because data you never wrote down cannot leak, cannot be
subpoenaed, and cannot be left in a backup for six years.

**Instructions.**

1. Say what you keep and for how long. A duration needs a number.
2. Say what happens on a deletion request, and how you would prove it happened.
3. Fill `never_store` with at least one real thing. This is the field that matters.

In [ ]:
policy = {
    "keep": "document versions and evaluation results",
    "keep_for": "",              # TODO(you): a duration with a number in it
    "delete_on_request": "",     # TODO(you): what happens, and how you prove it
    "never_store": [],           # TODO(you): what you refuse to keep at all
}
for k, v in policy.items():
    print(f"{k:20} {v if v else '(empty)'}")

**Expected output**

```
keep                 document versions and evaluation results
keep_for             18 months, then versions older than the current one are dropped
delete_on_request    Rows are deleted by doc_id within 30 days ...
never_store          ['the raw question text', 'any API key or authorization header', ...]
✅ d2-e3 passed
```

In [ ]:
check("d2-e3", policy)

## Review

The scorecard for this module. Every ❌ names the exercise and the hint.

In [ ]:
review("d2")